In [1]:
import os
os.environ ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ ["CUDA_VISIBLE_DEVICES"] = "1"

In [1]:
!nvidia-smi

Wed Apr 24 23:44:33 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.154.05             Driver Version: 535.154.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-PCIE-40GB          Off | 00000000:01:00.0 Off |                    0 |
| N/A   47C    P0              40W / 250W |    174MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [3]:
# !pip install chromadb

In [4]:
# pip install unstructured

In [5]:
# pip install InstructorEmbedding

In [ ]:
# pip install langchain
!pip install tokenizers==0.13.3

In [5]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN']=''
os.environ['OPENAI_API_KEY'] =''

In [6]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.embeddings import HuggingFaceHubEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI,HuggingFaceHub
from InstructorEmbedding import INSTRUCTOR
from transformers import pipeline

2023-09-25 21:32:29.843612: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-25 21:32:42.861903: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib64:
2023-09-25 21:32:42.862015: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/cuda/lib64:
2023-09-25 21:32:42.862024: W tensorflow

### Creating Chunks

In [4]:
loader=DirectoryLoader('')
docs=loader.load()
character_text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=250)
doc_texts=character_text_splitter.split_documents(docs)

## Creating Embeddings and saving in a database

In [ ]:
vectordb = Chroma.from_documents(documents=doc_texts, embedding=embeddings, persist_directory=persist_directory)

In [ ]:
vectordb.persist()
vectordb=None

## Loading existing OpenSource Embedding

In [4]:
embeddings = HuggingFaceInstructEmbeddings(
    query_instruction="Represent these legal documents for retrieval: ",
    model_name='hkunlp/instructor-large'
)
persist_directory = '/home/ramayana/ayush/Legal_QA/chroma_embedding_instruct'
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

load INSTRUCTOR_Transformer


Using embedded DuckDB with persistence: data will be stored in: /home/ramayana/ayush/Legal_QA/chroma_embedding_instruct


max_seq_length  512


## Loading existing OpenAI Embedding

In [ ]:
# pip install tiktoken

In [ ]:
# pip install openai

In [4]:
from langchain.embeddings.openai import OpenAIEmbeddings

In [7]:
persist_directory = '/home/ramayana/ayush/Legal_QA/chroma_legal_embedding_002'
EMBEDDING_MODEL='text-embedding-ada-002'
embeddings=OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

Using embedded DuckDB with persistence: data will be stored in: /home/ramayana/ayush/Legal_QA/chroma_legal_embedding_002


In [8]:
vectordb.as_retriever()

VectorStoreRetriever(vectorstore=<langchain.vectorstores.chroma.Chroma object at 0x7fe0bb86bf40>, search_type='similarity', search_kwargs={})

## OpenSource Model Trial-Flan T5

In [6]:
from transformers import pipeline
from langchain.llms.base import LLM
import torch


In [8]:
class customLLM(LLM):
    model_name = "google/flan-t5-base"
    pipeline = pipeline("text2text-generation", model=model_name, model_kwargs={"torch_dtype":torch.bfloat16})

    def _call(self, prompt, stop=None):
        return self.pipeline(prompt, max_length=9999)[0]["generated_text"]
 
    def _identifying_params(self):
        return {"name_of_model": self.model_name}

    def _llm_type(self):
        return "custom"


In [9]:
llm=customLLM()
chain=load_qa_chain(llm,chain_type='stuff')

In [10]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

## Flan Ul2 Trial


In [5]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch


In [ ]:
model = T5ForConditionalGeneration.from_pretrained("google/flan-ul2", device_map="auto", load_in_8bit=True)                                                                 
tokenizer = AutoTokenizer.from_pretrained("google/flan-ul2")


In [9]:
prompt='''
Answer the following question using the context by resoning step by step.If you don't know the answer,just say "Sorry,I dont know":\n\n
Question:{}\n\n
Context:{}
'''

In [14]:
query='what is punishment for murder under 18 age?'
context='Punishment for murder is death'

In [15]:
input_string=prompt.format(query,context)

In [17]:
inputs = tokenizer(input_string, return_tensors="pt").input_ids.to("cuda")
outputs = model.generate(inputs,max_length=1000)

result=tokenizer.decode(outputs[0])
result=result.lstrip('<pad>').rstrip('</s>').strip()

'Sorry,I dont know'

In [7]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [25]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

,Unnamed: 0,title,question,ground_truth,context
0,0,Submission of original documents to police for...,I have filed FIR via Magistrate. While submitt...,Dear Client When the police has asked you to b...,The delay in the despatch of the F.I.R. to the...
1,1,How to handle fake police threatening ?,How to handle fake police threatening ??,Dear Client In case you are receiving fake pol...,to do away with him in a fake encounter by coo...
2,2,"Case registered under 341, 323 & 325 IPC",My friend got served a notice from his local t...,"Dear Sir, It is called anticipatory bail and i...","(4) The time, place of arrest and venue of cus..."


In [27]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    context=ground_truth['context'][i].strip()
    input_string=prompt.format(query,context)
    inputs = tokenizer(input_string, return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(inputs,max_length=1000)
    result=tokenizer.decode(outputs[0])
    result=result.lstrip('<pad>').rstrip('</s>').strip()
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(2)

100%|██████████| 50/50 [11:00<00:00, 13.21s/it]


In [28]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

In [ ]:
from  transformers  import  AutoTokenizer, AutoModelWithLMHead, pipeline

In [ ]:
llm=HuggingFaceHub(repo_id='EleutherAI/gpt-j-6B')

In [ ]:
chain=load_qa_chain(llm,chain_type='stuff')

In [ ]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

In [ ]:
query='what is this?'
context_docs=vectordb.similarity_search(query)
result=chain.run(input_documents=context_docs,question=query)
print(result)

In [ ]:
# pip install openai

In [ ]:
llm=OpenAI(temperature=0,openai_api_key=os.environ['OPENAI_API_KEY'],model_name='text-davinci-003')
chain=load_qa_chain(llm,chain_type='stuff')

In [ ]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

In [ ]:
query='What is the procedure that the police must follow when they have been informed of a cognizable offence? Please provide relevant sections to bolster the answer.'
context_docs=vectordb.similarity_search(query)
result=chain.run(input_documents=context_docs,question=prompt+query)
print(result)

## LONGFORMER-BASE-4096 fine-tuned on SQuAD v1 trial

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [6]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [5]:
tokenizer = AutoTokenizer.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")
model = AutoModelForQuestionAnswering.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")

In [9]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

,Unnamed: 0,title,question,ground_truth,context
0,0,Submission of original documents to police for...,I have filed FIR via Magistrate. While submitt...,Dear Client When the police has asked you to b...,[29] For this part of the case the prosecution...
1,1,How to handle fake police threatening ?,How to handle fake police threatening ??,Dear Client In case you are receiving fake pol...,"Where it is solely a matter of threats, they m..."
2,2,"Case registered under 341, 323 & 325 IPC",My friend got served a notice from his local t...,"Dear Sir, It is called anticipatory bail and i...",(3) If such person is thereafter arrested with...


In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    context=ground_truth['context'][i].strip()
    encoding = tokenizer(query, context, return_tensors="pt")
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    # print(model(input_ids, attention_mask=attention_mask))
    output = model(input_ids, attention_mask=attention_mask)
    start_scores=output['start_logits']
    end_scores=output['end_logits']
    all_tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
    answer_tokens = all_tokens[torch.argmax(start_scores) :torch.argmax(end_scores)+1]
    result = tokenizer.decode(tokenizer.convert_tokens_to_ids(answer_tokens))
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(2)

In [23]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

## OpenAI QA Trial

In [29]:
from langchain.llms import OpenAI
from langchain.chains import LLMChain, ConstitutionalChain
from langchain.chains.constitutional_ai.models import ConstitutionalPrinciple
from langchain import PromptTemplate


In [27]:
llm = OpenAI(model_name='text-davinci-003',openai_api_key=os.environ['OPENAI_API_KEY'],temperature=0)

In [30]:
qa_prompt = PromptTemplate(
    template='''Your task is to answer a question as a legal assistant to the best of your abilities. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
Question:{query}
    ''',
    input_variables=["query"],
)

In [31]:
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


In [33]:
constitutional_chain = ConstitutionalChain.from_llm(
    llm=llm,
    chain=qa_chain,
    constitutional_principles=[
        ConstitutionalPrinciple(
            critique_request="Tell if this answer is good.",
            revision_request="Give a better answer.",
        )
    ],
)

In [35]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

,Unnamed: 0,title,question,ground_truth,context
0,0,Submission of original documents to police for...,I have filed FIR via Magistrate. While submitt...,Dear Client When the police has asked you to b...,[29] For this part of the case the prosecution...
1,1,How to handle fake police threatening ?,How to handle fake police threatening ??,Dear Client In case you are receiving fake pol...,"Where it is solely a matter of threats, they m..."
2,2,"Case registered under 341, 323 & 325 IPC",My friend got served a notice from his local t...,"Dear Sir, It is called anticipatory bail and i...",(3) If such person is thereafter arrested with...


In [37]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    result = constitutional_chain.run(query= query)
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(5)

100%|██████████| 50/50 [22:21<00:00, 26.82s/it]


In [38]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

## Testing QA System over Test Dataset

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [ ]:
ground_truth=pd.read_csv('')
ground_truth=ground_truth.drop(columns=['Unnamed: 0'])
ground_truth.head(3)

In [ ]:
!nvidia-smi

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query='Question:'+ground_truth['question'][i].strip()
    context_docs=vectordb.similarity_search(query)
    result=chain.run(input_documents=context_docs,question=prompt+query)
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['answer'][i],result.strip(),0])
    time.sleep(5)

In [ ]:
rows[5]

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)
f.close()